## 01 Exploratory Data Analysis

This notebook contains handling duplicates, properly implementing imputations, and creating multiple visualizations to increase our understanding in the dataset. 

In [1]:
#Load required libraries
import os
import polars as pl
import seaborn as sns
import matplotlib.pyplot as plt

#Get current working directory (.\\notebooks)
cwd = os.getcwd()

#Get parent folder to access (.\\Data)
parent = os.path.dirname(cwd)

#### File 1 of 11

In [2]:
#Get file path to load into environment
file_path_data_hourly = os.path.join(parent, "Data", "data_hourly.csv")

#Load data into environment using the file path variable with appropriate relative path, default encoding='utf-8'
data_hourly_df = pl.read_csv(file_path_data_hourly)

In [3]:
data_hourly_df.head(48)

ID,From,Date,Hour,Participation_Phase,Demand_kWh,Price_signal,Experiment_price_NOK_kWh,Temperature,Temperature24,Temperature48,Temperature72
str,str,str,i64,str,f64,str,str,f64,f64,f64,f64
"""Exp_43""","""2020-02-01T00:00:00Z""","""2020-02-01""",1,"""Phase_1""",0.348,null,"""NA""",7.6,6.1,5.0,4.4
"""Exp_43""","""2020-02-01T01:00:00Z""","""2020-02-01""",2,"""Phase_1""",0.589,null,"""NA""",8.2,6.2,5.1,4.4
"""Exp_43""","""2020-02-01T02:00:00Z""","""2020-02-01""",3,"""Phase_1""",0.358,null,"""NA""",8.4,6.3,5.3,4.5
"""Exp_43""","""2020-02-01T03:00:00Z""","""2020-02-01""",4,"""Phase_1""",0.582,null,"""NA""",8.4,6.4,5.4,4.5
"""Exp_43""","""2020-02-01T04:00:00Z""","""2020-02-01""",5,"""Phase_1""",0.364,null,"""NA""",8.2,6.5,5.5,4.6
…,…,…,…,…,…,…,…,…,…,…,…
"""Exp_43""","""2020-02-02T19:00:00Z""","""2020-02-02""",20,"""Phase_1""",2.825,null,"""NA""",1.2,2.5,4.5,4.9
"""Exp_43""","""2020-02-02T20:00:00Z""","""2020-02-02""",21,"""Phase_1""",2.837,null,"""NA""",0.9,2.4,4.4,4.9
"""Exp_43""","""2020-02-02T21:00:00Z""","""2020-02-02""",22,"""Phase_1""",2.099,null,"""NA""",0.6,2.3,4.3,4.9


In [4]:
data_hourly_df.describe()

statistic,ID,From,Date,Hour,Participation_Phase,Demand_kWh,Price_signal,Experiment_price_NOK_kWh,Temperature,Temperature24,Temperature48,Temperature72
str,str,str,str,f64,str,f64,str,str,f64,f64,f64,f64
"""count""","""13515600""","""13515600""","""13515600""",1.35156e7,"""13515600""",1.35156e7,"""1529208""","""13515600""",1.35156e7,1.35156e7,1.35156e7,1.35156e7
"""null_count""","""0""","""0""","""0""",0.0,"""0""",0.0,"""11986392""","""0""",0.0,0.0,0.0,0.0
"""mean""",null,null,null,12.5,null,2.56374,null,null,-0.322191,-0.330226,-0.340997,-0.354239
"""std""",null,null,null,6.922187,null,1.981698,null,null,5.107804,4.799844,4.662712,4.556716
"""min""","""Exp_1""","""2020-01-06T00:00:00Z""","""2020-01-06""",1.0,"""Phase_1""",0.0,"""A_10""","""0""",-21.8,-18.1,-17.3,-17.0
"""25%""",null,null,null,7.0,null,1.05,null,null,-3.8,-3.9,-4.0,-3.8
"""50%""",null,null,null,13.0,null,2.17,null,null,0.6,0.8,1.0,0.8
"""75%""",null,null,null,18.0,null,3.603,null,null,3.4,3.2,3.0,2.9
"""max""","""Exp_99""","""2021-03-26T23:00:00Z""","""2021-03-26""",24.0,"""Phase_2""",22.62,"""P_5""","""NA""",12.8,10.6,9.7,9.1


In [5]:
data_hourly_df.shape

(13515600, 12)

In [6]:
#print(type(data_hourly_df['ID']))
distinct_series = data_hourly_df['ID'].unique()
print(distinct_series)

print("\n Unique")
distinct_df = data_hourly_df.select(pl.col("ID").unique())
print(distinct_df)

shape: (4_429,)
Series: 'ID' [str]
[
	"Exp_259"
	"Exp_4559"
	"Exp_3606"
	"Exp_3867"
	"Exp_3849"
	…
	"Exp_2740"
	"Exp_283"
	"Exp_71"
	"Exp_521"
	"Exp_178"
]

 Unique
shape: (4_429, 1)
┌──────────┐
│ ID       │
│ ---      │
│ str      │
╞══════════╡
│ Exp_4788 │
│ Exp_5272 │
│ Exp_1903 │
│ Exp_440  │
│ Exp_52   │
│ …        │
│ Exp_212  │
│ Exp_40   │
│ Exp_3259 │
│ Exp_5404 │
│ Exp_2272 │
└──────────┘


In [7]:
# 11.9 million nulls in Price_signal column
print(data_hourly_df.select(pl.col("Price_signal").null_count()))

shape: (1, 1)
┌──────────────┐
│ Price_signal │
│ ---          │
│ u32          │
╞══════════════╡
│ 11986392     │
└──────────────┘


In [8]:
# Case-insensitive search for "Ipsos" in a Polars column
#ipsos_df = data_hourly_df.filter(pl.col("ID").str.contains("(?i)ipsos"))
#All Ipsos IDs were dropped for the actual experiment data

# To get the unique values directly:
distinct_ipsos = data_hourly_df.filter(
    pl.col("ID").str.contains("(?i)ipsos")
).select(pl.col("ID").unique())

print(distinct_ipsos)

distinct_exp = data_hourly_df.filter(
    pl.col("ID").str.contains("(?i)exp")
).select(pl.col("ID").unique())

print(distinct_exp)

shape: (0, 1)
┌─────┐
│ ID  │
│ --- │
│ str │
╞═════╡
└─────┘
shape: (4_429, 1)
┌──────────┐
│ ID       │
│ ---      │
│ str      │
╞══════════╡
│ Exp_3288 │
│ Exp_3868 │
│ Exp_3644 │
│ Exp_5129 │
│ Exp_2647 │
│ …        │
│ Exp_2549 │
│ Exp_3672 │
│ Exp_4419 │
│ Exp_2501 │
│ Exp_4394 │
└──────────┘


#### File 2 of 11

In [9]:
#Inspect each CSV file of the dataset, in turn
file_path_experiment_days = os.path.join(parent, "Data", "experiment_days.csv")
experiment_days_df = pl.read_csv(file_path_experiment_days)

In [10]:
experiment_days_df.head(5)

Participation_Phase,Price_signal,Date,Group
str,str,str,str
"""Phase_1""","""A_10""","""2020-02-13""","""H1"""
"""Phase_1""","""A_10""","""2020-03-02""","""H1"""
"""Phase_1""","""A_5""","""2020-02-14""","""H1"""
"""Phase_1""","""A_5""","""2020-03-03""","""H1"""
"""Phase_1""","""B_10""","""2020-03-04""","""H1"""


In [11]:
experiment_days_df.describe()

statistic,Participation_Phase,Price_signal,Date,Group
str,str,str,str,str
"""count""","""267""","""267""","""267""","""267"""
"""null_count""","""0""","""0""","""0""","""0"""
"""mean""",null,null,null,null
"""std""",null,null,null,null
"""min""","""Phase_1""","""A_10""","""2020-02-11""","""Ber_1"""
"""25%""",null,null,null,null
"""50%""",null,null,null,null
"""75%""",null,null,null,null
"""max""","""Phase_2""","""P_5""","""2021-03-25""","""Trond_1"""


In [12]:
experiment_days_df.shape

(267, 4)

#### File 3 of 11

In [13]:
file_path_participants = os.path.join(parent, "Data", "participants.csv")
participants_df = pl.read_csv(file_path_participants)

In [14]:
participants_df.head(5)

ID,Participation_Experiment,Recruitment_organisation,Municipality,Region,Survey1_answered,Survey2_answered,Survey3_answered,Participation_Phase,Control_Price_Phase1,Control_Price_Phase2,Group_Phase1,Group_Phase2,Participation_status_Phase1,Participation_status_Phase2
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""Exp_1""","""Yes""","""Transmission system operator""","""Bergen""","""Bergen""","""Yes""","""Yes""","""Yes""","""Both""","""Price group""","""Price group""","""H1""","""H1""","""OK""","""OK"""
"""Exp_2""","""Yes""","""Transmission system operator""","""Stavanger""","""Stavanger""","""Yes""","""No""","""Yes""","""Both""","""Price group""","""Price group""","""H1""","""H1""","""OK""","""OK"""
"""Exp_3""","""Yes""","""Transmission system operator""","""Oslo""","""Oslo""","""Yes""","""Yes""","""No""","""Both""","""Price group""","""Price group""","""H1""","""H1""","""OK""","""OK"""
"""Exp_4""","""Yes""","""Transmission system operator""","""Bergen""","""Bergen""","""Yes""","""Yes""","""Yes""","""Both""","""Price group""","""Price group""","""H1""","""H1""","""OK""","""OK"""
"""Exp_5""","""Yes""","""Transmission system operator""","""Bergen""","""Bergen""","""Yes""","""Yes""","""Yes""","""Both""","""Price group""","""Price group""","""H1""","""H1""","""OK""","""OK"""


In [15]:
participants_df.describe()

statistic,ID,Participation_Experiment,Recruitment_organisation,Municipality,Region,Survey1_answered,Survey2_answered,Survey3_answered,Participation_Phase,Control_Price_Phase1,Control_Price_Phase2,Group_Phase1,Group_Phase2,Participation_status_Phase1,Participation_status_Phase2
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""count""","""7410""","""7410""","""7410""","""7410""","""7410""","""7410""","""7410""","""7410""","""7410""","""7410""","""7410""","""7410""","""7410""","""7410""","""7410"""
"""null_count""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0"""
"""mean""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""std""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""min""","""Exp_1""","""No""","""Electricity supplier 1""","""Asker""","""Bergen""","""NA""","""NA""","""NA""","""Both""","""Control group""","""Control group""","""Control""","""Ber_1""","""NA""","""NA"""
"""25%""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""50%""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""75%""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""max""","""Ipsos_999""","""Yes""","""Transmission system operator""","""Trondheim""","""Trondheim""","""Yes""","""Yes""","""Yes""","""Phase_2""","""Price group""","""Price group""","""NA""","""Trond_Control""","""OK""","""OK"""


In [16]:
participants_df.shape

(7410, 15)

In [17]:
# There are 4,429 distinct ID in data_hourly, so some of these 7,410 ID from participants must have been dropped

distinct_series = participants_df['ID'].unique()
print(distinct_series)

shape: (7_410,)
Series: 'ID' [str]
[
	"Exp_555"
	"Exp_3219"
	"Exp_2381"
	"Exp_5202"
	"Exp_4962"
	…
	"Exp_3261"
	"Exp_3149"
	"Exp_1409"
	"Ipsos_615"
	"Exp_2993"
]


In [18]:
distinct_exp = participants_df.filter(
    pl.col("ID").str.contains("(?i)exp")
).select(pl.col("ID").unique())

print(distinct_exp)

shape: (5_410, 1)
┌──────────┐
│ ID       │
│ ---      │
│ str      │
╞══════════╡
│ Exp_3821 │
│ Exp_861  │
│ Exp_4554 │
│ Exp_2037 │
│ Exp_1214 │
│ …        │
│ Exp_2093 │
│ Exp_3924 │
│ Exp_477  │
│ Exp_2779 │
│ Exp_5179 │
└──────────┘


In [22]:
distinct_ipsos = participants_df.filter(
    pl.col("ID").str.contains("(?i)ipsos")
).select(pl.col("ID").unique())

print(distinct_ipsos)

shape: (2_000, 1)
┌────────────┐
│ ID         │
│ ---        │
│ str        │
╞════════════╡
│ Ipsos_956  │
│ Ipsos_1055 │
│ Ipsos_1195 │
│ Ipsos_1725 │
│ Ipsos_197  │
│ …          │
│ Ipsos_1538 │
│ Ipsos_605  │
│ Ipsos_1047 │
│ Ipsos_1788 │
│ Ipsos_1876 │
└────────────┘


In [23]:
# Extract all contiguous letter sequences from the ID column, flatten them, and get unique values
unique_designators = (
    participants_df.select(pl.col("ID")
        .cast(pl.String)  # Ensure the column is treated as strings
        .str.extract_all(
            r"[A-Za-z]+")  # Finds all alphabetical character blocks anywhere in the ID
    )
    .explode("ID")  # Flattens the lists of matches into individual rows
    .drop_nulls()  # Removes any nulls/empty matches
    .unique()  # Keeps only distinct designators
    .sort("ID")  # Sorts them alphabetically for easy reading
)

print(unique_designators)

shape: (2, 1)
┌───────┐
│ ID    │
│ ---   │
│ str   │
╞═══════╡
│ Exp   │
│ Ipsos │
└───────┘


In [36]:
#yes_count = (participants_df.filter(pl.col("Participation_Experiment").str.to_lowercase() == "yes").height)
#yes_count = len((participants_df.filter(pl.col("Participation_Experiment").str.to_lowercase() == "yes")))
yes_count = (participants_df.filter(pl.col("Participation_Experiment").str.contains(r"(?i)^yes$")).height)

print(f"Total 'Yes' count for Participation in the Experiment: {yes_count}")

Total 'Yes' count for Participation in the Experiment: 4429


#### File 4 of 11

In [37]:
file_path_price_signals = os.path.join(parent, "Data", "price_signals.csv")
price_signals_df = pl.read_csv(file_path_price_signals)

ComputeError: could not parse `5/10` as dtype `i64` at column 'Price_level' (column number 3)

The current offset in the file is 2355 bytes.

You might want to try:
- increasing `infer_schema_length` (e.g. `infer_schema_length=10000`),
- increasing `infer_schema_files` (e.g. `infer_schema_files=50`),
- specifying correct dtype with the `schema_overrides` argument
- setting `ignore_errors` to `True`,
- adding `5/10` to the `null_values` list.

Original error: ```invalid primitive value found during CSV parsing```

In [ ]:
price_signals_df.head(5)

In [ ]:
price_signals_df.describe()

In [ ]:
price_signals_df.shape

#### File 5 of 11

In [ ]:
file_path_survey1_answers = os.path.join(parent, "Data", "survey1_answers.csv")
survey1_answers_df = pl.read_csv(file_path_survey1_answers)

In [ ]:
survey1_answers_df.head(5)

ID,q1,q2,q3,q4,q5,q6,q6b1,q6b2,q6c,q7,q8,q9N1,q9N2,q9N3,q9N4,q9N5,q9N6,q9N7,q9N8,q9d1,q9d2,q9d3,q9d4,q10,q11,q12,q13,q14,q14b,q15_1,q15_2,q15_3,q15_4,q_ekstra_a,q_ekstra_b_1,q_ekstra_b_2,…,q16b5_1.1,q16b1_2,q16b2_2,q16b3_2,q16b4_2,q16b5_2.1,q16b1_3,q16b2_3,q16b3_3,q16b4_3,q16b5_3.1,q17,q17b.1,q17c,q18,q_kjonn,q19,q20.1,q20.2,q20.3,q20.4,q20.5,q20.6,q20.7,q20.8,q20.9,q21,q22,q23,q24.1,q24.2,q24.3,q24.4,q24.5,q24.6,q24.7,husstand
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,…,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str,i64,i64,i64,i64,i64,i64,i64,str
"""Exp_743""","""Yes""","""Variable price""","""No""","""Detached house""","""160 m2 or larger""","""Yes""","""Yes""","""Yes""","""NA""","""1981-1990""","""Yes""","""No""","""Additional heat source""","""Main heat source""","""No""","""Additional heat source""","""Additional heat source""","""No""","""No""","""Sometimes, mostly for having i…","""-""","""-""","""-""",20,"""Yes""","""Yes""","""Yes, manually (turn on and off…","""Own electric water heater""","""Yes, manually""","""-""","""-""","""What air ventilation system do…","""-""","""Yes""","""Electricity bill""","""-""",…,"""60""","""NA""","""NA""","""NA""","""NA""","""NA""","""NA""","""NA""","""NA""","""NA""","""NA""","""No""","""NA""","""NA""","""No""","""Woman""","""2 persons""",0,0,0,0,0,0,0,0,2,"""College / University, undergra…","""NOK 800 000 - 999 999""","""Yes""",0,0,0,0,0,2,0,"""-"""
"""Exp_236""","""Yes""","""Spot price / Hourly spot""","""No""","""Detached house""","""160 m2 or larger""","""Yes""","""No""","""NA""","""NA""","""1981-1990""","""Yes""","""Main heat source""","""Additional heat source""","""No""","""No""","""Additional heat source""","""No""","""No""","""No""","""Sometimes, mostly for having i…","""Only on the coldest days""","""-""","""-""",24,"""Yes""","""Yes""","""Yes, automated by time (set sc…","""Own electric water heater""","""No""","""What air ventilation system do…","""-""","""-""","""-""","""No""","""NA""","""NA""",…,"""NA""","""NA""","""NA""","""NA""","""NA""","""NA""","""NA""","""NA""","""NA""","""NA""","""NA""","""No""","""NA""","""NA""","""No""","""Man""","""3 persons""",0,0,0,0,0,1,0,0,2,"""College / University, higher d…","""NOK 1 000 000 - 1 499 999""","""Yes""",0,0,0,3,0,0,0,"""Single-family households with …"
"""Exp_744""","""Yes""","""Spot price / Hourly spot""","""No""","""Detached house""","""160 m2 or larger""","""Yes""","""Yes""","""Yes""","""NA""","""2011 or later""","""No""","""Main heat source""","""Additional heat source""","""Additional heat source""","""Additional heat source""","""Additional heat source""","""Additional heat source""","""Additional heat source""","""No""","""Sometimes, mostly for having i…","""-""","""-""","""-""",24,"""No""","""No""","""No""","""Own electric water heater""","""No""","""-""","""-""","""What air ventilation system do…","""-""","""Yes""","""-""","""-""",…,"""150""","""NA""","""NA""","""NA""","""NA""","""NA""","""NA""","""NA""","""NA""","""NA""","""NA""","""No""","""NA""","""NA""","""No""","""Man""","""3 persons""",0,0,0,0,1,0,0,2,0,"""College / University, higher d…","""NOK 1 500 000 or more""","""No""",0,1,0,2,0,0,0,"""-"""
"""Exp_745""","""Yes""","""Spot price / Hourly spot""","""Yes""","""Detached house""","""120-159 m2""","""Yes""","""No""","""NA""","""NA""","""1901-1920""","""Yes""","""Main heat source""","""Additional heat source""","""No""","""No""","""Additional heat source""","""No""","""No""","""No""","""-""","""-""","""-""","""Almost every day in winter""",20,"""Yes""","""Yes""","""Yes, automated by time (set sc…","""Own electric water heater""","""No""","""What air ventilation system do…","""-""","""-""","""-""","""No""","""NA""","""NA""",…,"""NA""","""NA""","""NA""","""NA""","""NA""","""NA""","""NA""","""NA""","""NA""","""NA""","""NA""","""No""","""NA""","""

In [ ]:
survey1_answers_df.describe()

statistic,ID,q1,q2,q3,q4,q5,q6,q6b1,q6b2,q6c,q7,q8,q9N1,q9N2,q9N3,q9N4,q9N5,q9N6,q9N7,q9N8,q9d1,q9d2,q9d3,q9d4,q10,q11,q12,q13,q14,q14b,q15_1,q15_2,q15_3,q15_4,q_ekstra_a,q_ekstra_b_1,…,q16b5_1.1,q16b1_2,q16b2_2,q16b3_2,q16b4_2,q16b5_2.1,q16b1_3,q16b2_3,q16b3_3,q16b4_3,q16b5_3.1,q17,q17b.1,q17c,q18,q_kjonn,q19,q20.1,q20.2,q20.3,q20.4,q20.5,q20.6,q20.7,q20.8,q20.9,q21,q22,q23,q24.1,q24.2,q24.3,q24.4,q24.5,q24.6,q24.7,husstand
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,f64,str,str,str,str,str,str,str,str,str,str,str,…,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str,str,f64,f64,f64,f64,f64,f64,f64,str
"""count""","""1682""","""1682""","""1682""","""1682""","""1682""","""1682""","""1682""","""1682""","""1682""","""1682""","""1682""","""1682""","""1682""","""1682""","""1682""","""1682""","""1682""","""1682""","""1682""","""1682""","""1682""","""1682""","""1682""","""1682""",1682.0,"""1682""","""1682""","""1682""","""1682""","""1682""","""1682""","""1682""","""1682""","""1682""","""1682""","""1682""",…,"""1682""","""1682""","""1682""","""1682""","""1682""","""1682""","""1682""","""1682""","""1682""","""1682""","""1682""","""1682""","""1682""","""1682""","""1682""","""1682""","""1682""",1682.0,1682.0,1682.0,1682.0,1682.0,1682.0,1682.0,1682.0,1682.0,"""1682""","""1682""","""1682""",1682.0,1682.0,1682.0,1682.0,1682.0,1682.0,1682.0,"""1682"""
"""null_count""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""",0.0,"""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""",…,"""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""0""","""0""","""0""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""0"""
"""mean""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,21.716409,null,null,null,null,null,null,null,null,null,null,null,…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0.027348,0.105232,0.332937,0.218787,0.177765,0.308561,0.497027,1.10107,0.940547,null,null,null,0.439952,0.176576,0.114744,1.095125,0.030916,0.604637,0.03805,null
"""std""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,2.492073,null,null,null,null,null,null,null,null,null,null,null,…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0.163145,0.385932,0.839094,0.881931,1.279,1.7279,2.748693,5.240715,5.957616,null,null,null,0.886288,0.50844,0.347383,1.029829,0.179881,1.950317,0.228234,null
"""min""","""Exp_1""","""No""","""Don't know""","""Don't know""","""Apartment block""","""100-119 m2""","""No""","""NA""","""NA""","""NA""","""1900 or earlier""","""No""","""Additional heat source""","""Additional heat source""","""Additional heat source""","""Additional heat source""","""Additional heat source""","""Additional heat source""","""Additional heat source""","""Additional heat source""","""-""","""-""","""-""","""-""",0.0,"""No""","""No""","""No""","""Electric flow heater""","""NA""","""-""","""-""","""-""","""-""","""No""","""-""",…,"""0""","""Charging station 32A, 3 phase …","""About once a month or less fre…","""Afternoon (1 pm - 5 pm)""","""NA""","""1""","""NA""","""NA""","""NA""","""NA""","""NA""","""No""","""-99""","""NA""","""No""","""Man""","""1 person""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""College / University, higher d…","""Below NOK 300 000""","""No""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""-"""
"""25%""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,21.0,null,null,null,null,null,null,null,null,nu

In [ ]:
survey1_answers_df.shape

(1682, 96)

#### File 6 of 11

In [ ]:
file_path_survey1_questions = os.path.join(parent, "Data", "survey1_questions.csv")
survey1_questions_df = pl.read_csv(file_path_survey1_questions)

In [ ]:
survey1_questions_df.shape

(255, 6)

#### File 7 of 11

In [ ]:
file_path_survey2_answers = os.path.join(parent, "Data", "survey2_answers.csv")
survey2_answers_df = pl.read_csv(file_path_survey2_answers)

#### File 8 of 11

In [ ]:
file_path_survey2_questions = os.path.join(parent, "Data", "survey2_questions.csv")
survey2_questions_df = pl.read_csv(file_path_survey2_questions)

#### File 9 of 11

In [ ]:
file_path_survey3_answers = os.path.join(parent, "Data", "survey3_answers.csv")
survey3_answers_df = pl.read_csv(file_path_survey3_answers)

ComputeError: could not parse `NA` as dtype `i64` at column 'QI7_1' (column number 6)

The current offset in the file is 523946 bytes.

You might want to try:
- increasing `infer_schema_length` (e.g. `infer_schema_length=10000`),
- increasing `infer_schema_files` (e.g. `infer_schema_files=50`),
- specifying correct dtype with the `schema_overrides` argument
- setting `ignore_errors` to `True`,
- adding `NA` to the `null_values` list.

Original error: ```invalid primitive value found during CSV parsing```

#### File 10 of 11

In [ ]:
file_path_survey3_questions = os.path.join(parent, "Data", "survey3_questions.csv")
survey3_questions_df = pl.read_csv(file_path_survey3_questions)

#### File 11 of 11

In [ ]:
import pyreadr

file_path_iFlex_dataset = os.path.join(parent, "Data", "iFlex_dataset.RData")
iFlex_dataset_df = pyreadr.read_r(file_path_iFlex_dataset, output_format="polars")

In [ ]:
print(iFlex_dataset_df.keys())

odict_keys(['data_hourly', 'participants', 'price_signals', 'experiment_days', 'survey1_answers', 'survey2_answers', 'survey3_answers', 'survey1_questions', 'survey2_questions', 'survey3_questions'])


In [ ]:
print(type(iFlex_dataset_df))

<class 'collections.OrderedDict'>


In [ ]:
iFlex_dataset_df['data_hourly']

ID,From,Date,Hour,Participation_Phase,Demand_kWh,Price_signal,Experiment_price_NOK_kWh,Temperature,Temperature24,Temperature48,Temperature72
str,datetime[μs],date,enum,enum,f64,enum,f64,f64,f64,f64,f64
"""Exp_43""",2020-02-01 00:00:00,2020-02-01,"""1""","""Phase_1""",0.348,"""""",null,7.6,6.1,5.0,4.4
"""Exp_43""",2020-02-01 01:00:00,2020-02-01,"""2""","""Phase_1""",0.589,"""""",null,8.2,6.2,5.1,4.4
"""Exp_43""",2020-02-01 02:00:00,2020-02-01,"""3""","""Phase_1""",0.358,"""""",null,8.4,6.3,5.3,4.5
"""Exp_43""",2020-02-01 03:00:00,2020-02-01,"""4""","""Phase_1""",0.582,"""""",null,8.4,6.4,5.4,4.5
"""Exp_43""",2020-02-01 04:00:00,2020-02-01,"""5""","""Phase_1""",0.364,"""""",null,8.2,6.5,5.5,4.6
…,…,…,…,…,…,…,…,…,…,…,…
"""Exp_5408""",2021-03-26 19:00:00,2021-03-26,"""20""","""Phase_2""",1.224,"""""",null,7.5,4.7,3.9,4.6
"""Exp_5408""",2021-03-26 20:00:00,2021-03-26,"""21""","""Phase_2""",1.668,"""""",null,7.3,4.9,4.0,4.6
"""Exp_5408""",2021-03-26 21:00:00,2021-03-26,"""22""","""Phase_2""",1.034,"""""",null,7.5,5.2,4.1,4.7
